# IDT -> YOLO Auto-Label Pipeline (Colab)

> Target scenario: floating waste on water, camera top-view at about 30 cm above water level.

This notebook:
1. installs and runs IDT to collect top-view floating-object style images
2. auto-labels with your YOLO model
3. builds train/valid/test folders
4. writes a ready `data.yaml`

Important: review labels before final training.

In [ ]:
import sys
import subprocess

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics', 'idt', 'pyyaml'], check=True)
print('Packages installed in current kernel:', sys.executable)

Packages installed in current kernel: /usr/bin/python3


## Optional: Mount Google Drive

In [ ]:
# Uncomment if needed
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# Optional: Upload local project zip/model directly to Colab runtime (/content)
# Use this when data is on your local PC (not Drive).

# Recommended upload files:
# - can_dataset.zip
# - plastic_bottle_dataset.zip
# - best.pt (optional, your trained model)

from pathlib import Path
import zipfile

TARGET_ROOT = Path('/content/most_recent_vrs')
TARGET_ROOT.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import files
    uploaded = files.upload()
    for fname in uploaded.keys():
        p = Path('/content') / fname
        low = p.name.lower()

        if p.suffix.lower() == '.zip':
            # Extract dataset zips directly to TARGET_ROOT for deterministic paths
            out_dir = TARGET_ROOT
            with zipfile.ZipFile(p, 'r') as zf:
                zf.extractall(out_dir)
            print(f'Extracted {p} -> {out_dir}')

            # If zip creates nested root folder, keep user informed
            can_exists = (TARGET_ROOT / 'can_dataset').exists()
            bottle_exists = (TARGET_ROOT / 'plastic_bottle_dataset').exists()
            print('After extract: can_dataset=', can_exists, ' plastic_bottle_dataset=', bottle_exists)
        else:
            # Move model file to TARGET_ROOT for easier discovery
            if low in {'best.pt', 'yolov8n.pt'}:
                dest = TARGET_ROOT / p.name
                p.replace(dest)
                print(f'Moved model to: {dest}')
            else:
                print(f'Uploaded file available at: {p}')
except Exception as e:
    print('Upload helper works in Colab notebooks only:', e)

In [ ]:
import os
import random
import shutil
import subprocess
import yaml
from pathlib import Path

# Auto-detect project root in Colab / Drive
PROJECT_ROOT_CANDIDATES = [
    Path('/content/most_recent_vrs'),
    Path('/content/drive/MyDrive/most recent vrs'),
    Path('/content/drive/MyDrive/most_recent_vrs'),
    Path('/content')
]

PROJECT_ROOT = None
for p in PROJECT_ROOT_CANDIDATES:
    if p.exists():
        PROJECT_ROOT = p
        break

if PROJECT_ROOT is None:
    PROJECT_ROOT = Path('/content/most_recent_vrs')
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    print('Created empty PROJECT_ROOT:', PROJECT_ROOT)
    print('Upload/copy your project files or mount Drive, then rerun this cell.')

# If can_dataset/plastic_bottle_dataset exist somewhere in /content, move PROJECT_ROOT there
if not (PROJECT_ROOT / 'can_dataset').exists() and not (PROJECT_ROOT / 'plastic_bottle_dataset').exists():
    found_can = list(Path('/content').rglob('can_dataset'))
    found_bottle = list(Path('/content').rglob('plastic_bottle_dataset'))
    found = [p for p in (found_can + found_bottle) if p.is_dir()]
    if found:
        # Use common parent if possible
        parent_counts = {}
        for p in found:
            parent_counts[p.parent] = parent_counts.get(p.parent, 0) + 1
        PROJECT_ROOT = sorted(parent_counts.items(), key=lambda kv: kv[1], reverse=True)[0][0]
        print('Adjusted PROJECT_ROOT based on discovered datasets:', PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
print('Working directory:', PROJECT_ROOT)

# Helpful diagnostics
for d in ['can_dataset', 'plastic_bottle_dataset', 'runs']:
    print(f"Exists {d}:", (PROJECT_ROOT / d).exists())

Working directory: /content/most_recent_vrs
Exists can_dataset: False
Exists plastic_bottle_dataset: False
Exists runs: False


## Step 1: Configure IDT dataset build

In [ ]:
# Top-view water scenario config (camera ~30 cm above water).
# Edit classes/keywords as needed for your exact project objects.
CLASSES = [
    {
        'CLASS_NAME': 'floating_can',
        'SEARCH_KEYWORDS': 'top view floating can on water, aluminum can floating in water, soda can floating water surface, overhead floating can'
    },
    {
        'CLASS_NAME': 'floating_bottle',
        'SEARCH_KEYWORDS': 'top view floating plastic bottle on water, bottle floating water surface overhead, floating PET bottle in pond top view'
    },
    {
        'CLASS_NAME': 'floating_trash',
        'SEARCH_KEYWORDS': 'top view floating trash on water, floating litter on water surface overhead, debris floating in water top view'
    }
]

IDT_CONFIG = {
    'DATASET_NAME': 'idt_raw_water_topview_30cm',
    'API_KEY': 'NONE',
    'SAMPLES_PER_SEARCH': 100,
    'IMAGE_SIZE': 640,
    'ENGINE': 'duckgo',
    'RESIZE_METHOD': 'longer_side',
    'VERBOSE': False,
    'CLASSES': CLASSES
}

with open('dataset.yaml', 'w', encoding='utf-8') as f:
    yaml.safe_dump(IDT_CONFIG, f, sort_keys=False)

print('Wrote IDT config to', PROJECT_ROOT / 'dataset.yaml')

Wrote IDT config to /content/most_recent_vrs/dataset.yaml


In [ ]:
# Build raw image dataset with IDT
# This creates: PROJECT_ROOT / IDT_CONFIG['DATASET_NAME'] / <class_name> / images
import sys
import subprocess
from pathlib import Path
import yaml
import shutil

yaml_path = Path('dataset.yaml')
if not yaml_path.exists():
    raise FileNotFoundError('dataset.yaml not found in current working directory')

# Compatibility fix for older IDT versions that require VERBOSE key
with open(yaml_path, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)
if 'VERBOSE' not in cfg:
    cfg['VERBOSE'] = False
    with open(yaml_path, 'w', encoding='utf-8') as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    print('Added missing VERBOSE key to dataset.yaml for IDT compatibility')

def run_cmd(cmd):
    print('\n$ ' + ' '.join(cmd))
    p = subprocess.run(cmd, text=True, capture_output=True)
    if p.stdout:
        print('\n[stdout]\n' + p.stdout[-6000:])
    if p.stderr:
        print('\n[stderr]\n' + p.stderr[-6000:])
    print('Return code:', p.returncode)
    return p.returncode

def collect_images(src_dir):
    exts = {'.jpg', '.jpeg', '.png', '.webp'}
    if not src_dir.exists():
        return []
    return [p for p in src_dir.rglob('*') if p.suffix.lower() in exts]

def find_named_dirs(dir_name):
    roots = [PROJECT_ROOT, Path('/content'), Path('/content/drive/MyDrive')]
    found = []
    for root in roots:
        if not root.exists():
            continue
        # direct candidate
        direct = root / dir_name
        if direct.exists() and direct.is_dir():
            found.append(direct)
        # shallow recursive search (reasonable in Colab)
        for p in root.rglob(dir_name):
            if p.is_dir() and p not in found:
                found.append(p)
            if len(found) >= 6:
                break
    return found

def fallback_build_raw_dataset(project_root, dataset_name):
    raw_root = project_root / dataset_name
    raw_root.mkdir(parents=True, exist_ok=True)

    can_dirs = find_named_dirs('can_dataset')
    bottle_dirs = find_named_dirs('plastic_bottle_dataset')

    class_sources = {
        'floating_can': can_dirs,
        'floating_bottle': bottle_dirs,
        'floating_trash': []
    }

    copied = 0
    per_class_limit = 1200

    print('Fallback source discovery:')
    print('  can_dataset dirs:', [str(p) for p in can_dirs])
    print('  plastic_bottle_dataset dirs:', [str(p) for p in bottle_dirs])

    for class_name in [c['CLASS_NAME'] for c in IDT_CONFIG['CLASSES']]:
        target_dir = raw_root / class_name
        target_dir.mkdir(parents=True, exist_ok=True)

        src_list = class_sources.get(class_name, [])
        class_images = []
        for src in src_list:
            class_images.extend(collect_images(src))

        # Remove duplicates while preserving order
        seen = set()
        unique_images = []
        for p in class_images:
            s = str(p)
            if s not in seen:
                seen.add(s)
                unique_images.append(p)

        for i, img_path in enumerate(unique_images[:per_class_limit]):
            out_name = f'{class_name}_{i:06d}{img_path.suffix.lower()}'
            shutil.copy2(img_path, target_dir / out_name)
            copied += 1

        print(f'Fallback copied {min(len(unique_images), per_class_limit)} images to {target_dir}')

    return copied

rc = run_cmd([sys.executable, '-m', 'idt', 'build'])

# Fallback for some environments where module entrypoint resolution differs
if rc != 0:
    rc = run_cmd([sys.executable, '-m', 'idt.__main__', 'build'])

if rc != 0:
    print('IDT build failed. Switching to local fallback dataset assembly...')
    copied = fallback_build_raw_dataset(PROJECT_ROOT, IDT_CONFIG['DATASET_NAME'])
    if copied <= 0:
        raise RuntimeError(
            'IDT build failed and fallback found no local images. '
            'Mount Google Drive and keep can_dataset/plastic_bottle_dataset inside your project folder, then rerun Cell 5, 7, 8.'
        )
    print(f'Fallback build completed with {copied} images')
else:
    print('IDT build completed')


$ /usr/bin/python3 -m idt build

[stdout]



                             
                8888888 8888888b. 88888888888 
                  888   888  "Y88b    888     
                  888   888    888    888     
                  888   888    888    888     
                  888   888    888    888     
                  888   888    888    888     
                  888   888  .d88P    888     
                8888888 8888888P"     888  
                                           
                        IMAGE DATASET TOOL V0.2                                 
                                                                                
                
Building idt_raw_water_topview_30cm dataset...

Creating floating_can class 

Creating floating_bottle class 

Creating floating_trash class 

Removing corrupt files
Dataset READY!

Return code: 0
IDT build completed


## Step 2: Auto-label with your YOLO model

In [ ]:
from ultralytics import YOLO
from pathlib import Path

# Prefer known paths first
MODEL_PATH_CANDIDATES = [
    PROJECT_ROOT / 'runs' / 'detect' / 'train2' / 'weights' / 'best.pt',
    PROJECT_ROOT / 'runs' / 'detect' / 'train' / 'weights' / 'best.pt',
    PROJECT_ROOT / 'best.pt',
    PROJECT_ROOT / 'yolov8n.pt',
    Path('/content/best.pt'),
    Path('/content/yolov8n.pt')
]

MODEL_PATH = None
for p in MODEL_PATH_CANDIDATES:
    if p.exists():
        MODEL_PATH = p
        break

# Recursive fallback search in Colab runtime + project folder
if MODEL_PATH is None:
    search_roots = [PROJECT_ROOT, Path('/content')]
    found = []
    for root in search_roots:
        if root.exists():
            found.extend(root.rglob('best.pt'))
            found.extend(root.rglob('yolov8n.pt'))
    if found:
        # Prefer best.pt over base yolov8n.pt
        found = sorted(found, key=lambda x: (x.name != 'best.pt', len(str(x))))
        MODEL_PATH = found[0]

# Final fallback: auto-download a base YOLO model so pipeline can proceed
if MODEL_PATH is None:
    print('No local model found. Downloading fallback model: yolov8n.pt')
    temp_model = YOLO('yolov8n.pt')  # triggers download in Colab if missing
    MODEL_PATH = Path(str(temp_model.ckpt_path)) if hasattr(temp_model, 'ckpt_path') else Path('/content/yolov8n.pt')

print('Using model:', MODEL_PATH)
model = YOLO(str(MODEL_PATH))

RAW_DATASET_DIR = PROJECT_ROOT / IDT_CONFIG['DATASET_NAME']
if not RAW_DATASET_DIR.exists():
    raise FileNotFoundError(f'Raw dataset directory not found: {RAW_DATASET_DIR}')

Using model: /content/most_recent_vrs/yolov8n.pt


In [ ]:
# Output dataset in YOLO detection structure
YOLO_OUT = PROJECT_ROOT / 'autolabel_yolo_dataset'
for split in ['train', 'valid', 'test']:
    (YOLO_OUT / split / 'images').mkdir(parents=True, exist_ok=True)
    (YOLO_OUT / split / 'labels').mkdir(parents=True, exist_ok=True)

# Split ratios
TRAIN_RATIO = 0.8
VALID_RATIO = 0.1
TEST_RATIO = 0.1
assert abs((TRAIN_RATIO + VALID_RATIO + TEST_RATIO) - 1.0) < 1e-9

# Auto-label settings for water-surface scene
CONF_THRES = 0.45
IOU_THRES = 0.5
MIN_BOX_AREA_NORM = 0.0008  # Filter tiny reflection/noise boxes
MAX_BOX_AREA_NORM = 0.65    # Filter unrealistic giant boxes
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.webp'}

def yolo_txt_from_boxes(result):
    lines = []
    h, w = result.orig_shape
    if result.boxes is None:
        return lines

    for b in result.boxes:
        cls_id = int(b.cls[0].item())
        x1, y1, x2, y2 = b.xyxy[0].tolist()
        bw_px = max(0.0, x2 - x1)
        bh_px = max(0.0, y2 - y1)
        area_norm = (bw_px * bh_px) / float(w * h)

        if area_norm < MIN_BOX_AREA_NORM or area_norm > MAX_BOX_AREA_NORM:
            continue

        xc = ((x1 + x2) / 2.0) / w
        yc = ((y1 + y2) / 2.0) / h
        bw = bw_px / w
        bh = bh_px / h
        lines.append(f'{cls_id} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}')
    return lines

def collect_images(src_dir):
    if not src_dir.exists():
        return []
    return [p for p in src_dir.rglob('*') if p.suffix.lower() in IMG_EXTS]

# 1) Primary source: IDT/fallback-assembled raw folder
all_images = collect_images(RAW_DATASET_DIR)

# 2) Recovery source: direct datasets if RAW_DATASET_DIR is empty
if not all_images:
    recovery_dirs = [
        PROJECT_ROOT / 'can_dataset',
        PROJECT_ROOT / 'plastic_bottle_dataset',
        Path('/content/can_dataset'),
        Path('/content/plastic_bottle_dataset')
    ]
    recovered = []
    for d in recovery_dirs:
        recovered.extend(collect_images(d))
    all_images = recovered

if not all_images:
    checked = [
        str(RAW_DATASET_DIR),
        str(PROJECT_ROOT / 'can_dataset'),
        str(PROJECT_ROOT / 'plastic_bottle_dataset'),
        '/content/can_dataset',
        '/content/plastic_bottle_dataset'
    ]
    raise RuntimeError(
        'No images found for auto-labeling. Checked: ' + '; '.join(checked) + '. '
        'Run upload cell, then rerun project-root cell and build cell.'
    )

# De-duplicate image paths while preserving order
seen = set()
unique_images = []
for p in all_images:
    s = str(p)
    if s not in seen:
        seen.add(s)
        unique_images.append(p)
all_images = unique_images

random.seed(42)
random.shuffle(all_images)

n_total = len(all_images)
n_train = int(n_total * TRAIN_RATIO)
n_valid = int(n_total * VALID_RATIO)

train_imgs = all_images[:n_train]
valid_imgs = all_images[n_train:n_train + n_valid]
test_imgs = all_images[n_train + n_valid:]

split_map = {
    'train': train_imgs,
    'valid': valid_imgs,
    'test': test_imgs
}

print(f'Total images: {n_total} | train={len(train_imgs)} valid={len(valid_imgs)} test={len(test_imgs)}')

for split, img_list in split_map.items():
    img_out_dir = YOLO_OUT / split / 'images'
    lbl_out_dir = YOLO_OUT / split / 'labels'

    for idx, img_path in enumerate(img_list):
        stem = f'{split}_{idx:06d}'
        out_img = img_out_dir / f'{stem}{img_path.suffix.lower()}'
        out_lbl = lbl_out_dir / f'{stem}.txt'

        shutil.copy2(img_path, out_img)

        pred = model.predict(source=str(out_img), conf=CONF_THRES, iou=IOU_THRES, verbose=False)[0]
        yolo_lines = yolo_txt_from_boxes(pred)

        with open(out_lbl, 'w', encoding='utf-8') as f:
            if yolo_lines:
                f.write('\n'.join(yolo_lines) + '\n')

print('Auto-labeling finished at:', YOLO_OUT)

RuntimeError: No images found for auto-labeling. Checked: /content/most_recent_vrs/idt_raw_water_topview_30cm; /content/most_recent_vrs/can_dataset; /content/most_recent_vrs/plastic_bottle_dataset; /content/can_dataset; /content/plastic_bottle_dataset. Run upload cell, then rerun project-root cell and build cell.

## Step 3: Write YOLO `data.yaml`

In [ ]:
names_dict = model.names if isinstance(model.names, dict) else {i: n for i, n in enumerate(model.names)}
class_names = [names_dict[i] for i in sorted(names_dict.keys())]

data_yaml = {
    'path': str(YOLO_OUT),
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': len(class_names),
    'names': class_names
}

yaml_path = YOLO_OUT / 'data.yaml'
with open(yaml_path, 'w', encoding='utf-8') as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

print('Wrote:', yaml_path)
print(data_yaml)

In [ ]:
# Quick stats
for split in ['train', 'valid', 'test']:
    img_count = len(list((YOLO_OUT / split / 'images').glob('*')))
    lbl_count = len(list((YOLO_OUT / split / 'labels').glob('*.txt')))
    print(f'{split}: images={img_count}, labels={lbl_count}')

print('Done. Next step: inspect labels visually before final training.')